# 01 数据下载

> **作业**：P02 金融数据获取、管理与初步分析  
> **姓名**：李泽欣  
> **日期**：2026-04-07

---

## 概述

本 Notebook 完成以下数据的下载：

| 数据类型 | 内容 | 来源 |
|---------|------|------|
| 个股行情 | 10 只 A 股股票过去 5 年后复权日度行情 | akshare / baostock |
| 市场指数 | 沪深 300、中证 500 日度数据 | baostock |
| 宏观指标 | CPI 同比、M2 同比增速 | akshare |
| 财务指标 | ROE、资产负债率 | akshare |

本次作业在 AI 辅助部分使用了 **两个模型**：

- **Claude Code**（Anthropic）
- **Codex**（OpenAI）

相关提示词、任务分工与对话摘要统一整理在 `03_analysis.ipynb` 的附录中，便于在最终提交时集中展示 AI 使用记录。

所有下载结果记录在 `download_log.txt` 中。


## 1. 环境准备与目录创建

使用 Python 代码自动创建项目目录结构（不手动新建）。

In [1]:
import os
import time
import datetime
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import baostock as bs
import akshare as ak

# 项目根目录
PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))

# 自动创建目录结构
dirs = [
    'data/stock',
    'data/index',
    'data/macro',
    'data/finance',
    'data/clean',
    'data/combined',
    'output',
]

for d in dirs:
    path = os.path.join(PROJECT_ROOT, d)
    os.makedirs(path, exist_ok=True)
    print(f'OK 已创建目录: {d}')

print()
print('目录结构创建完成！')

OK 已创建目录: data/stock
OK 已创建目录: data/index
OK 已创建目录: data/macro
OK 已创建目录: data/finance
OK 已创建目录: data/clean
OK 已创建目录: data/combined
OK 已创建目录: output

目录结构创建完成！


## 2. 定义股票列表与下载参数

选定 10 只股票，覆盖银行、汽车、房地产、白酒、能源、通讯、物流共 **7 个行业**。

In [2]:
# 股票信息字典：代码 -> (名称, 行业)
STOCK_INFO = {
    '000001': ('平安银行', '银行'),
    '601398': ('工商银行', '银行'),
    '002594': ('比亚迪',   '汽车'),
    '601633': ('长城汽车', '汽车'),
    '000002': ('万科A',    '房地产'),
    '600519': ('贵州茅台', '白酒'),
    '000858': ('五粮液',   '白酒'),
    '601857': ('中国石油', '能源'),
    '000063': ('中兴通讯', '通讯'),
    '002352': ('顺丰控股', '物流'),
}

# baostock 代码格式转换：000001 -> sz.000001, 600519 -> sh.600519
def to_bs_code(code: str) -> str:
    return f'sz.{code}' if code.startswith(('0', '3')) else f'sh.{code}'

# 指数信息
INDEX_INFO = {
    '000300': '沪深300',
    '000905': '中证500',
}

# 时间范围
START_DATE = '2020-01-01'
END_DATE   = '2026-04-01'

# 登录 baostock
lg = bs.login()
print(f'baostock 登录: {lg.error_msg}')

# 展示股票列表
stock_df = pd.DataFrame(
    [(code, name, industry) for code, (name, industry) in STOCK_INFO.items()],
    columns=['代码', '名称', '行业']
)
print()
print('选定的 10 只股票：')
display(stock_df)

login success!
baostock 登录: success

选定的 10 只股票：


,代码,名称,行业
0,000001,平安银行,银行
1,601398,工商银行,银行
2,002594,比亚迪,汽车
3,601633,长城汽车,汽车
4,000002,万科A,房地产
5,600519,贵州茅台,白酒
6,000858,五粮液,白酒
7,601857,中国石油,能源
8,000063,中兴通讯,通讯
9,002352,顺丰控股,物流


## 3. 下载日志工具

编写日志函数，将每次下载的结果记录到 `download_log.txt`。

In [3]:
LOG_FILE = os.path.join(PROJECT_ROOT, 'download_log.txt')

def log_download(category: str, name: str, status: str, detail: str = ''):
    """记录下载日志"""
    timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f'[{timestamp}] {status:<8} {category}_{name}  {detail}'
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(line + '\n')
    print(line)

# 清空旧日志
with open(LOG_FILE, 'w', encoding='utf-8') as f:
    f.write('')

print(f'日志文件已初始化: {LOG_FILE}')

日志文件已初始化: /Users/leezm/Desktop/金融数据/dshw-p01/download_log.txt


## 4. 下载个股行情数据

本次实际运行使用的是 `baostock` 的 `bs.query_history_k_data_plus()` 接口，而不是 `akshare`。下载字段包括 `date, open, high, low, close, volume, amount`，并统一附加股票代码列 `code`。

从实际运行结果看，10 只股票全部下载成功，每只股票都获得了 **1512 条日度记录**，时间区间覆盖 **2020-01-02 至 2026-04-01**。这说明当前数据源在所选股票样本上的稳定性较好，后续可以直接进入清洗和分析环节。


In [4]:
def download_stock_bs(code: str) -> pd.DataFrame:
    """使用 baostock 下载单只股票的后复权日度行情"""
    bs_code = to_bs_code(code)
    rs = bs.query_history_k_data_plus(
        bs_code,
        "date,open,high,low,close,volume,amount",
        start_date=START_DATE, end_date=END_DATE,
        frequency="d", adjustflag="1"  # 1=后复权
    )
    rows = []
    while (rs.error_code == '0') and rs.next():
        rows.append(rs.get_row_data())
    df = pd.DataFrame(rows, columns=rs.fields)
    if len(df) == 0:
        raise ValueError(f'{bs_code} 无数据返回')
    # 转数值类型
    for col in ['open', 'high', 'low', 'close', 'volume', 'amount']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['code'] = code
    return df


stock_data = {}
for code, (name, industry) in STOCK_INFO.items():
    try:
        df = download_stock_bs(code)
        save_path = os.path.join(PROJECT_ROOT, f'data/stock/stock_{code}.csv')
        df.to_csv(save_path, index=False, encoding='utf-8-sig')
        stock_data[code] = df
        log_download('stock', code, 'SUCCESS', f'shape={df.shape}  {name}({industry})')
    except Exception as e:
        log_download('stock', code, 'FAILED', f'Error: {e}')

print()
print(f'共成功下载 {len(stock_data)} 只股票数据')

[2026-04-07 15:00:55] SUCCESS  stock_000001  shape=(1512, 8)  平安银行(银行)
[2026-04-07 15:00:57] SUCCESS  stock_601398  shape=(1512, 8)  工商银行(银行)
[2026-04-07 15:00:59] SUCCESS  stock_002594  shape=(1512, 8)  比亚迪(汽车)
[2026-04-07 15:01:03] SUCCESS  stock_601633  shape=(1512, 8)  长城汽车(汽车)
[2026-04-07 15:01:05] SUCCESS  stock_000002  shape=(1512, 8)  万科A(房地产)
[2026-04-07 15:01:17] SUCCESS  stock_600519  shape=(1512, 8)  贵州茅台(白酒)
[2026-04-07 15:01:20] SUCCESS  stock_000858  shape=(1512, 8)  五粮液(白酒)
[2026-04-07 15:01:22] SUCCESS  stock_601857  shape=(1512, 8)  中国石油(能源)
[2026-04-07 15:01:27] SUCCESS  stock_000063  shape=(1512, 8)  中兴通讯(通讯)
[2026-04-07 15:01:30] SUCCESS  stock_002352  shape=(1512, 8)  顺丰控股(物流)

共成功下载 10 只股票数据


## 5. 下载市场指数数据

下载沪深 300 和中证 500 的日度数据，时间范围与股票一致。

- **沪深 300**（000300）：作为 CAPM 模型的市场基准
- **中证 500**（000905）：代表中小盘风格，与沪深 300 形成大小盘对比，有助于观察不同市值风格的收益差异

In [5]:
def download_index_bs(code: str) -> pd.DataFrame:
    """使用 baostock 下载指数日度数据"""
    bs_code = f'sh.{code}'
    rs = bs.query_history_k_data_plus(
        bs_code,
        "date,open,high,low,close,volume,amount",
        start_date=START_DATE, end_date=END_DATE,
        frequency="d"
    )
    rows = []
    while (rs.error_code == '0') and rs.next():
        rows.append(rs.get_row_data())
    df = pd.DataFrame(rows, columns=rs.fields)
    if len(df) == 0:
        raise ValueError(f'{bs_code} 无数据返回')
    for col in ['open', 'high', 'low', 'close', 'volume', 'amount']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


index_data = {}
for code, name in INDEX_INFO.items():
    try:
        df = download_index_bs(code)
        save_path = os.path.join(PROJECT_ROOT, f'data/index/index_{code}.csv')
        df.to_csv(save_path, index=False, encoding='utf-8-sig')
        index_data[code] = df
        log_download('index', code, 'SUCCESS', f'shape={df.shape}  {name}')
    except Exception as e:
        log_download('index', code, 'FAILED', f'Error: {e}')

print()
print(f'共成功下载 {len(index_data)} 个指数数据')

[2026-04-07 15:01:31] SUCCESS  index_000300  shape=(1512, 7)  沪深300
[2026-04-07 15:01:35] SUCCESS  index_000905  shape=(1512, 7)  中证500

共成功下载 2 个指数数据


## 6. 下载宏观经济数据

### 6.1 CPI 同比增速（必选）

CPI 是衡量通胀的核心指标，直接影响央行货币政策走向。高通胀环境下央行倾向于加息，不利于股市估值；低通胀/通缩则可能带来宽松政策预期。

### 6.2 M2 同比增速（自选）

M2 反映广义货币供应量的变化，是流动性的核心指标。M2 增速加快通常意味着货币政策宽松、市场流动性充裕，对股市形成正面支撑。

In [6]:
# --- 6.1 下载 CPI 数据 ---
try:
    cpi_df = ak.macro_china_cpi_yearly()
    # akshare 返回的 CPI 数据可能需要适配
    cpi_df.columns = [c.lower().strip() for c in cpi_df.columns]
    # 尝试识别日期列和数值列
    print('CPI 原始列名:', cpi_df.columns.tolist())
    print('CPI 前5行:')
    display(cpi_df.head())
    
    save_path = os.path.join(PROJECT_ROOT, 'data/macro/macro_cpi.csv')
    cpi_df.to_csv(save_path, index=False, encoding='utf-8-sig')
    log_download('macro', 'cpi', 'SUCCESS', f'shape={cpi_df.shape}')
except Exception as e:
    log_download('macro', 'cpi', 'FAILED', f'Error: {e}')
    print(f'CPI 下载失败: {e}')
    print('尝试备用接口...')
    try:
        cpi_df = ak.macro_china_cpi()
        save_path = os.path.join(PROJECT_ROOT, 'data/macro/macro_cpi.csv')
        cpi_df.to_csv(save_path, index=False, encoding='utf-8-sig')
        log_download('macro', 'cpi', 'SUCCESS', f'shape={cpi_df.shape} (备用接口)')
        print('CPI 前5行:')
        display(cpi_df.head())
    except Exception as e2:
        log_download('macro', 'cpi', 'FAILED', f'Error: {e2} (备用接口也失败)')
        print(f'备用接口也失败: {e2}')

CPI 原始列名: ['商品', '日期', '今值', '预测值', '前值']
CPI 前5行:


,商品,日期,今值,预测值,前值
0,中国CPI年率报告,1986-02-01,7.1,NaN,NaN
1,中国CPI年率报告,1986-03-01,7.1,NaN,7.1
2,中国CPI年率报告,1986-04-01,7.1,NaN,7.1
3,中国CPI年率报告,1986-05-01,7.1,NaN,7.1
4,中国CPI年率报告,1986-06-01,7.1,NaN,7.1


[2026-04-07 15:01:38] SUCCESS  macro_cpi  shape=(477, 5)


In [7]:
# --- 6.2 下载 M2 数据 ---
try:
    m2_df = ak.macro_china_money_supply()
    print('M2 原始列名:', m2_df.columns.tolist())
    print('M2 前5行:')
    display(m2_df.head())
    
    save_path = os.path.join(PROJECT_ROOT, 'data/macro/macro_m2.csv')
    m2_df.to_csv(save_path, index=False, encoding='utf-8-sig')
    log_download('macro', 'm2', 'SUCCESS', f'shape={m2_df.shape}')
except Exception as e:
    log_download('macro', 'm2', 'FAILED', f'Error: {e}')
    print(f'M2 下载失败: {e}')

M2 原始列名: ['月份', '货币和准货币(M2)-数量(亿元)', '货币和准货币(M2)-同比增长', '货币和准货币(M2)-环比增长', '货币(M1)-数量(亿元)', '货币(M1)-同比增长', '货币(M1)-环比增长', '流通中的现金(M0)-数量(亿元)', '流通中的现金(M0)-同比增长', '流通中的现金(M0)-环比增长']
M2 前5行:


,月份,货币和准货币(M2)-数量(亿元),货币和准货币(M2)-同比增长,货币和准货币(M2)-环比增长,货币(M1)-数量(亿元),货币(M1)-同比增长,货币(M1)-环比增长,流通中的现金(M0)-数量(亿元),流通中的现金(M0)-同比增长,流通中的现金(M0)-环比增长
0,2026年02月份,3492159.91,9.0,0.584687,1159258.82,5.9,-1.731121,151436.41,14.1,3.625196
1,2026年01月份,3471860.39,9.0,2.025077,1179680.52,4.9,2.123888,146138.60,2.7,3.452628
2,2025年12月份,3402948.06,8.5,0.980968,1155146.50,3.8,2.327986,141261.37,10.2,2.833230
3,2025年11月份,3369890.52,8.0,0.554356,1128866.64,4.9,0.795018,137369.38,10.6,1.395974
4,2025年10月份,3351312.31,8.2,-0.073312,1119962.73,6.2,-1.015713,135478.14,10.6,-0.246824


[2026-04-07 15:01:38] SUCCESS  macro_m2  shape=(218, 10)


## 7. 下载财务指标数据

获取 10 只股票最近 5 个年度的财务指标：
- **净资产收益率（ROE）**：衡量股东资本的盈利效率
- **资产负债率**：衡量企业的财务杠杆水平

将数据整理为**长格式**（Long format）：每行为一只股票一个年度的观测。

In [8]:
# 使用 ak.stock_financial_abstract() 获取财务摘要
# 该接口返回宽表：行=指标，列=报告期
# 经测试：第 11 行 = 净资产收益率(ROE)，第 16 行 = 资产负债率

finance_records = []

for code, (name, industry) in STOCK_INFO.items():
    try:
        df = ak.stock_financial_abstract(symbol=code)
        # 提取年报列（以 1231 结尾的列）
        year_cols = [c for c in df.columns[2:] if str(c).endswith('1231')]
        year_cols = sorted(year_cols, reverse=True)[:5]
        print(f'{code} {name}: 可用年度 = {year_cols}')

        # 提取 ROE（第 11 行）
        if len(df) > 11:
            for yc in year_cols:
                year = int(str(yc)[:4])
                val = df.iloc[11][yc]
                if isinstance(val, str):
                    val = val.replace('%', '')
                val = pd.to_numeric(val, errors='coerce')
                finance_records.append({
                    'code': code, 'name': name, 'industry': industry,
                    'year': year, 'indicator': 'ROE', 'value': val
                })

        # 提取资产负债率（第 16 行）
        if len(df) > 16:
            for yc in year_cols:
                year = int(str(yc)[:4])
                val = df.iloc[16][yc]
                if isinstance(val, str):
                    val = val.replace('%', '')
                val = pd.to_numeric(val, errors='coerce')
                finance_records.append({
                    'code': code, 'name': name, 'industry': industry,
                    'year': year, 'indicator': '资产负债率', 'value': val
                })

        log_download('finance', code, 'SUCCESS', f'{name} 年报{len(year_cols)}个年度')
        time.sleep(2)
    except Exception as e:
        log_download('finance', code, 'FAILED', f'Error: {e}')
        print(f'{code} {name} 下载失败: {e}')

finance_df = pd.DataFrame(finance_records)
save_path = os.path.join(PROJECT_ROOT, 'data/finance/finance_ratios.csv')
finance_df.to_csv(save_path, index=False, encoding='utf-8-sig')

print()
print(f'财务数据已保存，共 {len(finance_df)} 条记录')
print()
print('长格式数据预览：')
display(finance_df.head(10))

000001 平安银行: 可用年度 = ['20251231', '20241231', '20231231', '20221231', '20211231']
[2026-04-07 15:01:39] SUCCESS  finance_000001  平安银行 年报5个年度
601398 工商银行: 可用年度 = ['20251231', '20241231', '20231231', '20221231', '20211231']
[2026-04-07 15:01:42] SUCCESS  finance_601398  工商银行 年报5个年度
002594 比亚迪: 可用年度 = ['20251231', '20241231', '20231231', '20221231', '20211231']
[2026-04-07 15:01:44] SUCCESS  finance_002594  比亚迪 年报5个年度
601633 长城汽车: 可用年度 = ['20251231', '20241231', '20231231', '20221231', '20211231']
[2026-04-07 15:01:47] SUCCESS  finance_601633  长城汽车 年报5个年度
000002 万科A: 可用年度 = ['20251231', '20241231', '20231231', '20221231', '20211231']
[2026-04-07 15:01:49] SUCCESS  finance_000002  万科A 年报5个年度
600519 贵州茅台: 可用年度 = ['20241231', '20231231', '20221231', '20211231', '20201231']
[2026-04-07 15:01:52] SUCCESS  finance_600519  贵州茅台 年报5个年度
000858 五粮液: 可用年度 = ['20241231', '20231231', '20221231', '20211231', '20201231']
[2026-04-07 15:01:55] SUCCESS  finance_000858  五粮液 年报5个年度
601857 中国石油: 可用年度 = ['2025

,code,name,industry,year,indicator,value
0,000001,平安银行,银行,2025,ROE,9.150000
1,000001,平安银行,银行,2024,ROE,10.080000
2,000001,平安银行,银行,2023,ROE,11.380000
3,000001,平安银行,银行,2022,ROE,12.360000
4,000001,平安银行,银行,2021,ROE,10.850000
5,000001,平安银行,银行,2025,资产负债率,90.698536
6,000001,平安银行,银行,2024,资产负债率,91.422796
7,000001,平安银行,银行,2023,资产负债率,91.546121
8,000001,平安银行,银行,2022,资产负债率,91.831647
9,000001,平安银行,银行,2021,资产负债率,91.964692


## 8. 下载日志查看

查看所有数据下载的完整日志。

In [9]:
print('=' * 70)
print('下载日志 (download_log.txt)')
print('=' * 70)
with open(LOG_FILE, 'r', encoding='utf-8') as f:
    print(f.read())
print('=' * 70)

下载日志 (download_log.txt)
[2026-04-07 15:00:55] SUCCESS  stock_000001  shape=(1512, 8)  平安银行(银行)
[2026-04-07 15:00:57] SUCCESS  stock_601398  shape=(1512, 8)  工商银行(银行)
[2026-04-07 15:00:59] SUCCESS  stock_002594  shape=(1512, 8)  比亚迪(汽车)
[2026-04-07 15:01:03] SUCCESS  stock_601633  shape=(1512, 8)  长城汽车(汽车)
[2026-04-07 15:01:05] SUCCESS  stock_000002  shape=(1512, 8)  万科A(房地产)
[2026-04-07 15:01:17] SUCCESS  stock_600519  shape=(1512, 8)  贵州茅台(白酒)
[2026-04-07 15:01:20] SUCCESS  stock_000858  shape=(1512, 8)  五粮液(白酒)
[2026-04-07 15:01:22] SUCCESS  stock_601857  shape=(1512, 8)  中国石油(能源)
[2026-04-07 15:01:27] SUCCESS  stock_000063  shape=(1512, 8)  中兴通讯(通讯)
[2026-04-07 15:01:30] SUCCESS  stock_002352  shape=(1512, 8)  顺丰控股(物流)
[2026-04-07 15:01:31] SUCCESS  index_000300  shape=(1512, 7)  沪深300
[2026-04-07 15:01:35] SUCCESS  index_000905  shape=(1512, 7)  中证500
[2026-04-07 15:01:38] SUCCESS  macro_cpi  shape=(477, 5)
[2026-04-07 15:01:38] SUCCESS  macro_m2  shape=(218, 10)
[2026-04-07 15:01:

## 9. 数据概览

从本次真实运行结果看，股票与指数数据都比较完整：

- 10 只个股全部下载成功，每只股票 `shape=(1512, 8)`
- 2 个指数全部下载成功，每个指数 `shape=(1512, 7)`
- CPI 原始数据 `shape=(477, 5)`，M2 原始数据 `shape=(218, 10)`
- 财务数据最终整理为 **100 条长格式记录**，对应 10 只股票 × 5 个年度 × 2 个指标

财务数据在年份上存在轻微不一致：大多数股票可取得 2021-2025 年报数据，而贵州茅台和五粮液当前抓取到的是 2020-2024 年的 5 个年度。这种差异不会影响“最近 5 个年度”的作业要求，但在横向比较时需要注意年份口径并不完全一致。


In [10]:
print()
print('=== 财务数据概览 ===')
print(f'总记录数: {len(finance_df)}')
if len(finance_df) > 0:
    print(f'指标类型: {finance_df["indicator"].unique().tolist()}')
    print(f'年份范围: {finance_df["year"].min()} ~ {finance_df["year"].max()}')
else:
    print('暂无财务数据（下载失败）')


=== 财务数据概览 ===
总记录数: 100
指标类型: ['ROE', '资产负债率']
年份范围: 2020 ~ 2025


In [11]:
# 退出 baostock 连接
bs.logout()
print('baostock 已退出')

logout success!
baostock 已退出


---

## 小结

本 Notebook 已经基于真实运行结果完成了四类数据下载，并成功写入项目目录：

1. **个股行情**：10 只 A 股后复权日度数据，全部成功，保存至 `data/stock/`
2. **市场指数**：沪深 300 与中证 500，均成功下载，保存至 `data/index/`
3. **宏观数据**：CPI 同比与 M2 同比，保存至 `data/macro/`
4. **财务数据**：ROE 与资产负债率，整理为长格式后保存至 `data/finance/finance_ratios.csv`

下载日志显示，本次运行中所有任务均为 `SUCCESS`，没有失败记录。整体来看，数据获取阶段已经满足作业对“股票、指数、宏观、财务”四类数据的完整性要求，为后续的数据清洗、可视化和 CAPM 回归分析提供了可靠基础。
